# Games, stages, and transparent lifting

A small executable sketch of the [game algebra](../docs/design/game-algebra.md). The stage and lifting machinery comes from `shapiq`; the three-player table is a notebook-local example. Follow the trace to see exactly when each stage runs.

In [1]:
# Copyright (c) 2026 shapiq contributors
from __future__ import annotations

from dataclasses import dataclass
from typing import ClassVar

import jax
import jax.numpy as jnp
from jaxtyping import Float, Int, Shaped, jaxtyped
from ndlike.protected_axis.jax import JaxAxisProtected

from shapiq.coalition import FiniteCoalition
from shapiq.game import Game, PGame
from shapiq.partial import Partial, defer, delay, partial
from shapiq.players import Players
from shapiq.transformation import lifted

In [2]:
@dataclass
class BitPlayers(Players[FiniteCoalition], JaxAxisProtected[jax.Array]):
    ids: jax.Array
    protected_axes: ClassVar[dict[str, int]] = {"ids": 1}

    def is_valid(self, coalition, *, batch_shape=None):
        if not isinstance(coalition, jax.Array):
            return False
        if coalition.dtype != jnp.bool_ or coalition.ndim == 0:
            return False
        if coalition.shape[-1] != self.ids.shape[-1]:
            return False
        shape = self.shape if batch_shape is None else batch_shape
        try:
            if jnp.broadcast_shapes(self.shape, shape) != shape:
                return False
            jnp.broadcast_shapes(shape, coalition.shape[:-1])
        except ValueError:
            return False
        return True

In [3]:
@dataclass
class TableGame(Game[FiniteCoalition, jax.Array], JaxAxisProtected[jax.Array]):
    values: jax.Array  # batch axes, then one value per coalition
    _players: BitPlayers  # one scalar player set shared by this demo's games
    protected_axes: ClassVar[dict[str, int]] = {"values": 1}

    def __post_init__(self):
        with jaxtyped("context"):
            assert isinstance(self, Shaped[Game, "*batch"])
            assert isinstance(self.players, Shaped[Players, "#*batch"])
            assert isinstance(self.players.ids, Int[jax.Array, "#*batch n"])
            assert isinstance(self.values, Float[jax.Array, "*batch 2**n"])

    @property
    def players(self):
        return self._players

    def eval(self, coalition):
        if not self.is_valid(coalition):
            msg = "Inadmissible coalition batch"
            raise ValueError(msg)
        index = jnp.sum(coalition * (1 << jnp.arange(self.players.ids.size)), axis=-1)
        shape = jnp.broadcast_shapes(self.shape, index.shape)
        table = jnp.broadcast_to(self.values, (*shape, self.values.shape[-1]))
        index = jnp.broadcast_to(index, shape)
        return jnp.take_along_axis(table, index[..., None], axis=-1)[..., 0]

In [4]:
coalitions = jnp.array(
    [
        [0, 0, 0],  # empty
        [1, 0, 0],  # {0}
        [0, 1, 0],  # {1}
        [1, 1, 0],  # {0, 1}
        [0, 0, 1],  # {2}
        [1, 0, 1],  # {0, 2}
        [0, 1, 1],  # {1, 2}
        [1, 1, 1],  # {0, 1, 2}
    ],
    dtype=bool,
)
g = TableGame(jnp.array([[0.0, 1.0, 2.0, 4.0, 3.0, 5.0, 6.0, 10.0]]), BitPlayers(jnp.arange(3)))
print("game batch:", g.shape, "shared players:", g.players.shape)
for coalition, value in zip(coalitions.tolist(), g(coalitions).tolist(), strict=True):
    print(coalition, "->", value)

game batch: (1,) shared players: ()
[False, False, False] -> 0.0
[True, False, False] -> 1.0
[False, True, False] -> 2.0
[True, True, False] -> 4.0
[False, False, True] -> 3.0
[True, False, True] -> 5.0
[False, True, True] -> 6.0
[True, True, True] -> 10.0


## A transformation flowing through a family of families

`@lifted` lets the same handler accept a game or any nested family. `make_family(g)` has two stages: scale factors, then an offset. Return annotations (or explicit `result_type=`) identify each stage's result. This sketch exposes eventual batch metadata as `.shape`.

In [5]:
trace = []


@lifted(output_shape=lambda shape: shape)
def double_game(game: TableGame) -> TableGame:
    trace.append("double")
    return TableGame(2 * game.values, game.players)


def make_family(game):
    assert game.shape == (1,)  # this toy creator starts with a one-game batch

    def bind_factors(*, factors: jax.Array) -> Partial[..., TableGame]:
        trace.append("bind factors")

        def bind_offset(*, offset: float) -> TableGame:
            trace.append("materialize")
            values = jnp.asarray(factors)[:, None] * game.values + offset
            return TableGame(values, game.players)

        return partial(bind_offset, shape=(len(factors),))

    return partial(bind_factors, shape=(None,))

In [6]:
trace.clear()
direct = double_game(g)
assert trace == ["double"]
assert bool(jnp.allclose(direct.eval(coalitions), 2 * g(coalitions)))
trace.clear()
graph = double_game(make_family(g))
print("unbound:", graph.shape, trace)
assert graph.shape == (None,) and trace == []

remaining = graph.bind(factors=jnp.array([1.0, 2.0]))
print("one stage bound:", remaining.shape, trace)
assert isinstance(remaining, PGame) and remaining.shape == (2,)
assert trace == ["bind factors"]

result = remaining(offset=0.5)
print("fully bound:", result.shape, trace)
assert result.shape == (2,) and trace == ["bind factors", "materialize", "double"]

# Add a singleton batch axis to evaluate every coalition for every game.
values = result[:, None](coalitions)
print(values)
assert bool(jnp.allclose(values, 2 * (jnp.array([1.0, 2.0])[:, None] * g.values + 0.5)))

unbound: (None,) []
one stage bound: (2,) ['bind factors']
fully bound: (2,) ['bind factors', 'materialize', 'double']


[[ 1.  3.  5.  9.  7. 11. 13. 21.]
 [ 1.  5.  9. 17. 13. 21. 25. 41.]]


## Explicit deferral and evaluation

`delay` suspends a computation; `defer` suspends an existing value. A zero-parameter call executes that stage. `.eval(S)` on a partial records evaluation, so the whole pipeline uses the same spelling.

In [7]:
def construct() -> TableGame:
    trace.append("construct")
    return g


def explain(game):
    return double_game(game).eval(coalitions)


trace.clear()
pending_values = explain(delay(construct, shape=g.shape))
assert trace == []
values = pending_values()
assert trace == ["construct", "double"]
assert bool(jnp.allclose(values, 2 * g(coalitions)))
print("deferred values:", values, trace)

# Ordinary map transforms one result, including a result that is itself partial.
mapped = defer(g).map(double_game)
assert bool(jnp.allclose(mapped().values, direct.values))

deferred values: [ 0.  2.  4.  8.  6. 10. 12. 20.] ['construct', 'double']


## A handler whose input is itself partial

This handler replaces the innermost `offset` stage with a `baseline` stage. Given three stages, lifting maps over the outer two: `lifted(t)(pppg) = pppg.map(lambda pp: pp.map(t))`. The new stage is the handler's result, retained intact.

In [8]:
@lifted(output_shape=lambda shape: shape)
def replace_offset(pg: Partial[..., TableGame]) -> Partial[..., TableGame]:
    trace.append("replace offset")

    def by_baseline(*, baseline: float) -> TableGame:
        return pg(offset=baseline)

    return partial(by_baseline, shape=pg.shape)


trace.clear()
tower = defer(make_family(g))
changed = replace_offset(tower)
assert changed.shape == (None,) and trace == []
after_thunk = changed()
assert trace == []
after_factors = after_thunk(factors=jnp.array([1.0, 2.0]))
assert after_factors.shape == (2,)
assert trace == ["bind factors", "replace offset"]
print("new stage awaits baseline:", after_factors.shape, trace)
result = after_factors(baseline=0.5)
assert trace == ["bind factors", "replace offset", "materialize"]
assert bool(jnp.allclose(result.values, jnp.array([1.0, 2.0])[:, None] * g.values + 0.5))
print("result:", result.values)

# Evaluation also traverses the full nested family.
pending = double_game(make_family(g)).eval(coalitions[0])
assert bool(jnp.allclose(pending(factors=jnp.array([1.0, 2.0]))(offset=0.5), 1.0))

new stage awaits baseline: (2,) ['bind factors', 'replace offset']
result: [[ 0.5  1.5  2.5  4.5  3.5  5.5  6.5 10.5]
 [ 0.5  2.5  4.5  8.5  6.5 10.5 12.5 20.5]]
